# 03 — Inference
Score patients in two ways: with the saved pipeline directly, and through the deployed REST API.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)

In [2]:
import json
from src.predict import load_model, predict
model = load_model()
patient = json.loads((ROOT / 'sample_request.json').read_text())
predict(model, [patient])

[{'prediction': 1,
  'label': 'disease',
  'probability_disease': 0.9478,
  'confidence': 0.9478}]

In [3]:
df = pd.read_csv(ROOT / 'data' / 'processed' / 'heart_clean.csv')
batch = df.sample(5, random_state=1)
out = pd.DataFrame(predict(model, batch.drop(columns='target').to_dict('records')))
out.insert(0, 'actual', batch['target'].values)
out

,actual,prediction,label,probability_disease,confidence
0,0,0,no_disease,0.3486,0.6514
1,0,0,no_disease,0.4732,0.5268
2,0,0,no_disease,0.2226,0.7774
3,1,1,disease,0.9184,0.9184
4,1,0,no_disease,0.1884,0.8116


### Through the API (Docker Compose on :8010, Kubernetes on :8020 / Helm on :8030)

In [4]:
import requests
for port in (8010, 8020, 8030):
    try:
        r = requests.post(f'http://localhost:{port}/predict', json=patient, timeout=5)
        print(port, r.status_code, r.json())
    except Exception as e:
        print(port, 'unreachable:', type(e).__name__)

8010 200 {'prediction': 1, 'label': 'disease', 'probability_disease': 0.9478, 'confidence': 0.9478, 'model_version': '1.0.0'}


8020 200 {'prediction': 1, 'label': 'disease', 'probability_disease': 0.9478, 'confidence': 0.9478, 'model_version': '1.0.0'}


8030 200 {'prediction': 1, 'label': 'disease', 'probability_disease': 0.9478, 'confidence': 0.9478, 'model_version': '1.0.0'}


In [5]:
r = requests.post('http://localhost:8010/predict', json={**patient, 'age': 500}, timeout=5)
r.status_code, r.json()['detail'][0]['msg']

(422, 'Input should be less than or equal to 120')